# ☁️ Dia 80 — Construindo um grafo de acesso "quem alcança o quê"

**Investigador Cyber 90 Dias** · Fase 6 · Rank Jutsu `A`

> Baseado no *Day 80* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-6-cloud-ai-system-investigation/day-80.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A Atende+ diz que já resolveu: *"desativamos a conta do dev.externo"*. Rafa empurra um post-it: *"E a Ana?"*

Neste notebook você vai:

1. **Montar o grafo do GCP** com o `gcp_graph.py` do original, comentado.
2. **Mudar a decisão `READS`** por formulário e ver Lee Chen aparecer.
3. **Testar os cortes A e B** e gerar a tabela de leitores.
4. **Grafo da AWS** com as quatro arestas `CreateAccessKey`.
5. **Quiz: qual aresta cortar?** e o grafo da Lia.

🐍 Só biblioteca padrão do Python. Os dados do caso Blue Harbor vêm do GitHub do original; sem rede, o notebook usa a cópia embutida (com o mesmo SHA-256). Os comandos `jq` do README viram Python equivalente aqui. Rode as células de cima para baixo (Shift+Enter).

## 1. Evidência com custódia

In [ ]:
# ------------------------------------------------------------
# CARREGADOR DE EVIDÊNCIAS (offline-first, com custódia)
# 1. Tenta baixar cada arquivo do repositório original no GitHub.
# 2. Confere o SHA-256 contra o valor esperado (o hash do upstream).
# 3. Se a rede falhar OU o hash não bater, usa a cópia embutida aqui.
# ------------------------------------------------------------
import base64          # a cópia embutida está em base64 (texto puro)
import gzip            # ...e comprimida com gzip, para o notebook não ficar enorme
import hashlib         # SHA-256: a impressão digital de cada arquivo
import json            # os logs de nuvem são JSON
import urllib.request  # baixar da internet sem instalar nada

BASE_RAW = "https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-6-cloud-ai-system-investigation/resources/"

# Hash esperado de cada arquivo (calculado na cópia do upstream deste repositório).
HASH_ESPERADO = {
    "case-blueharbor/gcp/iam-project-policy.json": "ba06f54e053dbe3661bb31490a020aa5142e958566cf499597c10db6c3010ac7",
    "case-blueharbor/gcp/iam-reporting-sa-policy.json": "b6e163a76a2aee374611020535ed8bc878c755ed1ec835b4b9aef378237daaa2",
    "case-blueharbor/gcp/iam-bucket-customer-exports.json": "7f2bd90183b00fc5fb36cc2c195c087c2f8b3bd188fe0f80b28f499bc8359797",
    "case-blueharbor/gcp/groups-export.csv": "3b5b3666f9af782dae68d22e0a5c02fd62982c17c68524cbd5db6a972697c516",
    "case-blueharbor/gcp/service-accounts.csv": "b113a7498497c296e41050e9ff81b13f79893c94a99e241b84bbab43abd7284e",
    "case-blueharbor/aws/iam-authorization-details.json": "ed25c4c9a9b4867e4c499752f639497dd9c1be109d857124c91d2b1b307a1726",
    "case-blueharbor/aws/cloudtrail-lookup-events.json": "820d3aa67653c7e4588192097195a54697f8158d8af4434ba5990a39909f5926",
}

# Cópia embutida (plano B offline): o arquivo inteiro, gzip + base64.
_AMOSTRA = {
    "case-blueharbor/gcp/iam-project-policy.json": "H4sIAAAAAAAC/72UWU/CQBDH3/kUTZ/pKuIFiQkoPJigiXeMMWbbDsvKdqfuwRHCd3dbjoBitIT40na7/5n5zezsTEqe51Mbc3OBssuZ9uvei/vneZP8udjtIPsqWBXlQoHsfpyCE/jN1tXl9dttu9nyl4rp/Ou1vPCsQQ14lBtQIe5mKz2zyNS50g+5jLncRJZAEoJaJ1o4bUYRWmnqOupBbAWoQNNGKCz0qApRBVRSMTY80oTThLC5FZ1ZkQgTv/yjTwUpKuOYCvv8VgKFIs8/e+s9J0mtAcKlNlS6gHHCJRlU5iUpF0//sHJaPTnYr9SOjquNSKCN5wJdMOeIBzGkAsc7TRhca6HaPj1t0+woghDNTrkoTwU1XVQJsc66ACBTaNN6TA0NQLIVKAIjmqQuyEqNM991AUBck8pN2l8wQ84+LKgxyeI9chjuGPWv4d8xfChWpp3d0l8QM7P1WAVB8yOKHQfBPnUNsUWZ3Fxkblg43GyAPilu/psAh3KLzsiLr43eIuBgtReXkxwMZZnmfPjcq/ZvaqNO+yy/Dv7AIXCUbrNSmpY+AUAHzyCVBgAA",
    "case-blueharbor/gcp/iam-reporting-sa-policy.json": "H4sIAAAAAAAC/5WQPw/CIBDF934KwtzWP2MTE+vq7GCMw0FPJCnQHLRqjN9dbGvUwUQXyN3j93h314QxLrSttFWeF2wXa8au/RkVg0YgvYS+qci1TVFBgAytWoq6xSOQcJTjGUxTIx/f7tOnD7nYLYbbTzSY3CN1WmIppWttWOOlrIy2A3lLf0jxaVAQNo5CnCLz8BYpAwv1JWjp88evaqRgoHLpDE+/enp5xKqtkf72/HsBm1iOwycjxzGAejCr0/YwhXIm5nLRp+Vd3IZ2Noqz5JbcAXGO8P3EAQAA",
    "case-blueharbor/gcp/iam-bucket-customer-exports.json": "H4sIAAAAAAAC/7WRPW/CMBCG9/wKyzNJ90hIhKpzJQYWxHB2TsYl9kVnB4QQ/x3nAzFAJVq1iy373sf32D5nQkhlfW29CbIUm7QW4jyMqeLQKeR7Ydhsmb5Qx4/aRuJSNR3ugBVxDh6aU7Q6yNlD+vPo8ZvwlN3eIMnUYGo5zOEtpC5gsGjQgD4tO73H8bQRvMxeV15bPP6hxQqh/pFGQD5YjZXW1PlYMrbEMb18HmDxTKqw4AozUTBShSb3qiyp/tZV7az/vWXo2l4zVxT/UXL8mskymyiJEUxPvFf7ucwu2RVMF79crwIAAA==",
    "case-blueharbor/gcp/groups-export.csv": "H4sIAAAAAAAC/53QQQrCMBAF0L1niTlDEbqsQkVclon9tIEkEyapmNsr2I1QW3Q1iz88/swgPMUOnqxTHt5A5tHlEqGEHXY9ZdojDJVxE0YSw6LxIB8dVBRbSAeyspReznWrTtdj3a4imb02FLNNGV+Zpm4OG04irwUFqboj9Cz7OdA39h8IBXIl5bSEOLz2R4StImvGLwelKUaW/FeV92uf3FDPAcQBAAA=",
    "case-blueharbor/gcp/service-accounts.csv": "H4sIAAAAAAAC/5WQsW4CMQyGd57CI0g5dFCpUrdKTF06lAeIfIk5IpL45Dht7+0besDOZkv+v882JQzR+FCmiLPNmMigKrozeatsaiGxCTOOrb/QXOxAJxay+37/avs3u9uvXOg8TZHn9yFWOqMMLF2LxFmDK9uAaTs2zHdwhM5xzbp1nMzhA5ZYMaaH9Q/LJTJ6CJ6yBp3hRJ4ENXDerIQmFg157Ao+p/kM41njDK4W5UQC9HtFAWYPxxdIQYTFLPxOas4k3Q7WrdKQCG5EuCE3pl+V63NqbHPP7nKIXJv1nodFB46zCkdzZdfpf5GB9Tn0cQkClhKKYlYY0F0o+5v0q+bHLXfJY7aJ/wBUinz1CAIAAA==",
    "case-blueharbor/aws/iam-authorization-details.json": "H4sIAAAAAAAC/+1Y23LiOBB9z1e4eJyJB9sQkvjNa5MsO4CJIcnUpPIgbAW8JSxWkpNlp/j3lWxz8RWcpGZ2aoeqpIi71WqdPn3U8bcTSWrcUkgsyICP+j5lDV164E8l6Vv0m9tHgM3500azcbp5JJYMwQKKxx4IgAy8hR+k7T1PWI2eZfxxY/zW/zpuG60b6/bsvHOzczRIILwACXTwQnUfLHRd5R+Nf1r8o4c8UrNoC5NAwKDFf0QATdHasqLJqjpRO7qm6Erro8J/K7sF1wSHy80JH3cZMAbcOfQGfI8Z9EYY+a4P6RaGfShiOITHanN6Q2TFgxLAMDFcF1K63XLPveic4ssysjaLomyDrJNvu5wnYBYl+Bg9WJ/WqtczDDxMZAoWZfUyP5vdu/FNx9S6X4zBqN+tW6+iLXL16sjKmay0J2pLVzS93a6s1xaMTfouDjhaLodri9Rj6jgx7rn1FaW8iyKPIXoaQ/Lsu/AzXBUW08JuuICBCLwfThIhCPVxEJ9Q5XRUZPU8FYI7jRkHIVn/kDJJmXixux8VxoHA64HFFSYTDswUQTrHmPnBLBM+WtN9eoIui+qJEH4pcjFcFmf6kLNxqyjsNWQfClYmVoHsh0bO+liwlQMpDokbgZxbsj49HgIskLNfglxh3vPYMVHjJuQblSNgQQSPcrxdeqmItUE70G55TFN/P54UWfKqUqGEeekpbihxOp4vfgkgSXfOHUDhvvzAv8FiieAnFy8KlO41ukafXZnAJSappshIWyJo4zvT6Y5sZ6LWlbaSXYpuo46sXk6US72t6qr2nW6jBfgHB+PWVYjQ6y+jfJAj7qJKQgSYA1PMBwRnwF1JC58QTKRwiTDwIDmVOMgIuNCTpitpEBnvic8gcTCCEgf4jN/178QcjDwZMnSAM3bf6k76iqLU5kwufo4tLVm5kFvKRNX0iCdvZ8tbL0OerwyfIVmxefaO+SG3YB1pbxRdWwduoVcpZmYOO0lscblqjtTRmsyMtj/kpB0Tdl6PNuy86w4t2xnXIudMRGoe3CtNnnfTKhPh0LsHzJ2L0cYO0OrVilUaqkwftpUSalKzUGLJ5gxZYUp7JUVy7E2RBj3HsZ2aEkJ4pGb5RgalvA3Fs4q2PNiSpe2Yae3qNmyMiB+4/hKgnChw4xXkws53iVABrovDgNFPM4xn2TmgYC7c627KqL479b3P5vdw2vN47j7LTmENEween6zMZTRmPN9Z968QIFpglwrT1EEYnUBV2u0LTb1UWueqenbW0bQLVanWlXWhrmzUZp0iT33Jjq9QeRkyGfMW+Lk0u3Acpy19FDJ7+qcI8+qpmUfR9SkfN+aATKPRU4xvVI7xaiZ/fo8pOiM+OcXcyc7R7xgS10RqhiPDsAa9YW88cYyJ7Rim+aVaag68hDitEEILPoEQsYQ5cQbP+8oWISHoYooe4mZ1a+vR2Ar4P7LcwEgIMydKwlY3wP9lBkmPziWAJ8Cmy5KAe+ygfOx/E3naDYyv9pD73vb7nHXdGrSr2uUX695FRUve49CWjCNxlRFYTD2gv8sLnf8ajQ8Omjkym3371ro3JubvTtew7OHxZD5ir1+UfjulXQHzi4BZtyB1iT+FJRTf8yx/n7nnFL/WLPRCeEYrgkTmXTY/ZyOJ2eRkffIvxIJtXhgaAAA=",
    "case-blueharbor/aws/cloudtrail-lookup-events.json": "H4sIAAAAAAAC/+2dW3PbRrKA3/MrWMrLbq1HmvsF5+Xguqu1FTuWk+zZ463UYGYgs0IROiRlr+PKfz8DiBIlgRRhmZQBL5QKLRHXRnd/6Jnp6fn03Wh0kL5308X8IBj9r/9rNPpUf15/f2z9hgOotXbKCGALYQHNNQGaQwswp5zBnOWF4wfP7hz4gz531aEvxvNFdGl+c/4SN3u8dtq+nE4+VjssZpdutSU0xs3nz93HqwuHz49DkP4jPHn1IgU/vHwDXqfhi3tXejO+uhKGmAOoAMJvIA4wDSD7C4QBhPf2Py0vZ6Y+Yk4O9bn+vZzqD/NDU56vdvxp7mbTpQTz9wbM3EU5W4ynZ7dlmNcnqp/cv26+jiflpX0z0+NJfbHqBJ/eHrjq95/dbD4up28PgrcH6BDBtwfP3h5c+isdW791vPjot/h9Fx8vXL3PcXhS3Ue928VsPDXjCz05tvW28DgJl8/l9Of4dfrq5es3qN5Tz66u4P8NvFzBWJ8HAfI/2P8Q/xNUlzy6I9XVgcaUl5XGr27w1hHXm681c3UH61RzI1Kl/nq3e9f549nyYVRaq3dYo7Z/1qdxK2VdnemetlY73VzslrFd3fOH+Wt3dv3ML+fA6fkCXD2nK+0dvwqtnXnBroRW8pBVqoGHmNzIEp75a1w90w9zYCbjI3yIxCGFo3N7FJWL0pQz9z06JOxQji613wpH5fxoMp5e/vt7figP6x31zLz7/t+S/8rpaKKnZ0cXHxfvyun35BDhQzbyAp3rqT26L8LM/d+lmy9e6ZkXcuEtyN/I9HIyqTfNL8rp3KUTd1658O0t9UHHSX3XkKo4YrHgIeM0ScLVg7veoYV7L2/mym/9UZXb3ujy2mLDD/PwYhzryaTe3Qukz+p7q31hddTMeVse+6/Ch22uPnusF+6snH2sdzm5OaW3pYPa6f54tpFb3CBOmGVAMMIBZRgDnSMIiBKIM0GlkGwDt04X5cWL8uzsns+vuFXoyXwv4CIBRVvAZSrILCrIDADrGsAq9a0H2CatrQHZLevrK8jui7AOZN5YptciX6t79ZSCGznvqr/eeJRPLt07PcvLGai/uFJNWybGGUIJyTCFIUtCJBtMbIOO+0yskfCVoVjtMJuVs7i0y8vXZp+46djZ1eYT/50/pt6j8tFg9DnuNhrPR9NyMdKXXvOz8e/Ojhbl6MLNinJ2HoxuqfCWFYxyZ7Q/lz9yNF4CA+R67g++KCdj83HkH1H5YT5avHObTqHNonaC7ehnChWIawVy4bz+bI78O01o4AqmCwslx3gT+v/qFpW2Vlh72sCVBAhuC1wX8wH83QM/ghsi1/vqWkP8htH1lfvrBdlBGItSglOFEYpjmtIIN5DdxuV7GcYqaxByuQCISQNoJaLOZQ6EYppS66ikeAPLYi/rwt3A6dGh7M/pD8nL1zde8zz9n3YwwwHEW2Dm3bwNzPxB1r/r5/q8SySLn8fpz6c/8hgvH83nkGwl0qMx1tRLE2S3L7OVYpXC1lOsoac1FLtnbn1l2DoxNsSv214Za+j26ZYu7/259Q01X+jF5XwZ2C3G793W95bfbGpxEv//VePCXYwQfjaqND9aqnwUnvj7/aMBXUUTqEIuUArTiEjegG4bNnUxTm7RecA5YtoH/8Q5BqgzHGhcKOCQxbZgRUEkeaDT88Z6Ht3v+VjmooDAgbk9Ym6lsMcz966x9bnf854UuyPu2niSKpFhDBOZJjgiWDS7AFoAoJfxpEQcKasgoEJr/+Es0EwbAGWuFRcOG2YeIttioc07ZysnfVW13MfuKzBu6+jOwLhuMW7TuE5rxq0xu17TbpM8e+YekyGlTCZZSKI0Ec2uzzZ46CX3CKJcWiIByY2PVYkTQMuiAF5gp5jUyDL6APcqRT056GBAh2CuT6CrFPZloKvtrM9kWwmwg26/kHGZJCgRqWBRmKEGrtp4dT9HrzlRhFnPYQUpoLDAlWAMOMJyrDmW0BT7y7p5PLDQbvJuBl49Ea/QkHCz24SbBDPBM4wkxyxGgq1pWW737A4ia19jyyt73TKwPCfBLXV91oDyvUPbDyRTI2AuGANEFNLrivrXC5UYGKck1c4Rw/c+kPxIFiMVULmboeQBxk8B41phwxjy/saQI8xgjCJEUZRS3wxukLmNt/cymMyNIFBxAhgxGNDq5aOxgEBSpRkj0gm7KYU7nM8vz93rcuJ+GS/e/eLyh3H28GhyC3DBgMkA4d2A66aTxHvgf9/KGfMPb/JxMTbzw6pZduaPeD82bgmBu6fbP+duPdONvFve2vzwrCzPJq66xeDRwjUx9WWnun79vpqV7/3vs013vJbPXpP3+Kxri7Ng5m3u6GRcBRW/zMbe5SsjPPqCW30YwEvDezyAN3rKI0HsIXroMcvgGgxXzCVHS+C2ZfBD/YvVww63vjrXquTqtP6XU3/vXsAdWZW9nOkqTDt1ppza6jYJh/Bz+jpFiglNWRyrNKY8Shq8b4PFfg5f04Jjy3zzghWF9pJ5GXWB/Z9MwCqp37HcbY5br+Lkulv6UVHr6fF1kl8S/hBC/4O2kR+9qfrOUADJTroPrH9aQNvz8bRLEevffwyjF/88pSH5MfnJtxJ//JyIdSXSoyLWNTppvgduX+MBWq6U9ei+g3tW9kWI5OsIeVL+Pp5M9BHzGPzTiTbj6aKcv/uv0fF04SYj/8Xo5enoHyMEf0XsV/HnUXhx4cHt8ufjxRGH7NDrm43+9Pxvb05ePBtNxr+50V+d+a3882hpFEc+/uWjU13o2fjmgF1FqhgREkYxjjGJ0zBtdnu28e+eTtohUEvDAZOQAEpJAbSXBjhoKOeKCWs3ZTsmbm5m49zVvjx/Sm4RFmCx01k7A7/2zq9aaV88V+eu0Q0Yu5tkQ1msmBQJzTAjMFwzz2a7t/czyUbkiBmngM0xBFRjC7RQEliplXTEGWPy/YzePJZhiAcUD7FXP9hVK+srjtt8q8BKZMS4SETGFGIkamYFtvHrXgILI8oNyiEopCw8ibEBGlkOKMZKCSaRk3Bzg/HELWZjk+iFfrpBDvQGqQD5l7jaQq3zcjpelB4JZ8NYRwfGOlZ6Ww+wTepa34hcWd4XoWw5GPANoQzFIcJcpBHBUYxYc8JcG4/vJcoMxrkrjAIF0RTQggmguTc+wqWGBDJGnOpQ7FV5Aw4YG2KvHsRe18oaYq+dA4vHqahARTIBEcmamclt/LqfjUVisPQfAOkCAWqdt0Umqx49CYXVyiJt9pGZ/FhcQRUwtZu05IFXe+dVra2vl5P8zeIqS0TKJEeYJQqtmUDWxqt7iSuWI2KpgcCyXHjBcAE0yR3wjWEpCOKESNSl+EpWA/tkGFfsBa+Wyhriq903CGWSRGmIUpr4/6K0WUGlhV/3c+aX0QyxwgJUOAuo9G1eDXkOcm1sQXPLLX0gGSIu54twan+qspqfElpYBHhrKUA3QKsT0KqVtWEw0bVK3b1lZQO47qZxYU5FlEGcpAomBDbngLXw7342DC0l3PnGLkZIAoq9TFqoHChNrYTaEUR1tyItzAM8RFp9gValrCHS2jmwlArjOImoSolkWaaaTcMWft3PUURc9d1BAQpXGECRUUCbQoAcIlNIZHWRi+70ZImAke0Jp0NPVjd4tdTW0JO1c1yJWJGIkihKKCOwWeWtjVf3EleUW20wJADnTleCIaCZYSDnSjiMKXIUbkk2DSd6dj5/0qwHUaUuQjpkPfQs62Gpty/OerhrekPaw93M+QjiKEuTSEHFY9WEWRuf72faA2KUFogD7OoCyIT6VjDzQaW1RlmuNetWt7yoRqagGhqL/Qi+amUNjcWdA0umHFLGJcyUJBFpTlJs49e9BJY1oqASSpBTZQEtclV121EAEYWEMS1ygbsYfUH/IpdD9NW76KvW2xB97TUpgkjOIJUxRJzDKG7ArI3P97OrXrJqCjsHHBtSVUKHoDJqQCilOXOykBJ2a94iYgHD3imGeYt9CsSWShvmLe4TY55eIQ+VhxmOUsaaJdzaeHs/YzJOCKbKAMytF4yI4mpeuSZKUIuqohiiezEZCwgPMB5isp7FZEu9DTHZPmGWqYSmJPPhGMlwuGYB2DY+30uYCaN9m9lRgJjNq3rmxgvmf/PPGRIuqRJEdWc0kgWY7WztliEO23scVmtrGI3cw2hkFhIeo4hDFNFm6Zs2Xt3PIo3YWguZBhJLCag0DOicCsAtxEoggwxHW2Kv4+l8oafGPSm2EA3IttjLGTxgqxPYqrW1HlsNJT0Qat1Y2oCvuwsWJCohPGGcxFEG42aN2TZe3s9pQcoJiyTykvjgkSosgUayALkuqKKF0wKrjvWA0eolvnWJ0qEHrFMIWypt6AHb62QhTjHFPpwimMgYiTXLLW/39l5irAojkZEEeBBDQJ12VX5IXq0ASHOHiNPKblpu2T/ZcuJelGd3nX9X1bFr44e+5bG1rP/4bDqeDrRqQatHQ6jWw4Z8iDWPf91yybfM5T8XP5/uP4hqjeLL2jGuV7a7ky4hME04Z4xJLDyjGmBq47+tazov7+zUK/R42pEUe868KIoBipHyUkEOvEQGFIxr5olsmDVdm8zovQVv758fJjN2JL6qlTVMZtxLYBWlcaJoREOexZnia5LtW/h3LwMrhRC0hCKAFSsAhVoAXRAMlC2QY4JQlRddWIMELZeCwHBYg2RYg+QJ1yBZGd6wBsk3swYJjzPOsyhNZMpQvKYMYxss9nMNkoL5VxYRCnAofCTOpARaYwIcpZQJ6QTVuGPjGbAqrtyG/cN4RgeC1ZW2hvGMvU1PqKrwy8gHpZ5jiWxWZWzj5f1MhYPM5DSHwDDuA3GCq44ExAHDUBNKcgIV6xy+kAqYCCgb8NUHfF1ra8DX3lYg8dCiWRpCHiHJcLMURxsv72cpDqGo04IAha0EVFsEtKMEGM9qSrTlDJGOJL/VjkBJANmQ/NYTbNXaGpLfdo6rmCmaZhBmSRhnYXPiQRuv7ieuIM8LKTAQWhFAC8d8GGkh8OEj1Rw6bpHs2MSD2hGgCPBQiqNXEw9WehsmHuwVZkRKxBglIedSyawJsxY+38+6QlrBnOQFgAhWNUa0AloWOYCC+KaxgMIi053YSwa4RX3sIfbqRuy11NYQe+08c5chmaQw44onOObNMrNtvLqfa+66XBQGFsAp7gCFlHvBpAVYQO0kL6BArHuxV9VxEjAyxF49i72Wehtir/2u/hYjRDOsMhjHKlqz8m4Ln+/npE9NWVFwBKDNq8XsqgUMpIIgVwpzRRGmuelMGbTaHaq1qOlQBq0PwddSWUMZtN2PM1JFJckoSmDMEkia0z5b+HUvgQWdFUIVFCimbVVLxIeVCBngLObYGM0szrs3zlgXBGRyGGfsCbdqbQ3jjPvCF4MsSlNGMsnDDK5ZDa6Nl/cSXyi3EgkCQQEJBdSgakV0pIFUUngxCVHcdmnaJ6wmQVMaQD5M++wTwpZKG6Z97rUPLEQRT5VCaYYzGjZnV7Xx9p4W30Ck8DYMclQJpp0BmhUYMCqosEYxWujOYayqQgMHjPULY7XSBoztE2OEZhCFEoYxE5JQvKYIx3Zv72c0JoxURBUAC5cDCnFVmBIhgPOCMYYsLDYudfm1MEYDUqVBDhjrE8aWShswtte5ojwjvkkZZkkSwRA2EyjaeHs/54paYoQuFJA293zOqzLh0osoNOYUWuhbzvnmSe5XPayvysnYfHxSjuGAoKEjvyf8qpT16I78e1Y2gOsuuCBDGU2gTFJGE9YcfWzj3/3szFfOaWQogMSHk5Rx7gNLXQBNhYauUJ7L+VeqHgSXVWsQGaoHfc3qQSs9DNWDnrp6EE4yCYWISASxSFAzx6uN//a5elBuhCAwZ1VL10eMUudA45q+HrqUSIEl60IRDrishYDUUIRjKMLxhEU4VoY3FOH4ZopwqFRRKUOhJGY+KG0WXWqDxS4X4fCf//ruj+/+H/URPUXS2wAA",
}

EVIDENCIA = {}   # nome do arquivo -> bytes
ORIGEM = {}      # nome do arquivo -> "github" ou "embutida"

for nome in HASH_ESPERADO:
    try:
        # timeout curto: no Colab sem rede (ou num teste offline) não queremos travar
        with urllib.request.urlopen(BASE_RAW + nome, timeout=15) as r:
            dados = r.read()
        if hashlib.sha256(dados).hexdigest() != HASH_ESPERADO[nome]:
            # O arquivo mudou no GitHub: não é a evidência que recebemos. Não usamos.
            raise ValueError("hash diferente do esperado")
        ORIGEM[nome] = "github"
    except Exception as erro:
        dados = gzip.decompress(base64.b64decode(_AMOSTRA[nome]))
        ORIGEM[nome] = f"embutida ({type(erro).__name__})"
    EVIDENCIA[nome] = dados

# Recibo de custódia: confere TODOS os hashes de novo, depois de carregados.
for nome, dados in EVIDENCIA.items():
    ok = hashlib.sha256(dados).hexdigest() == HASH_ESPERADO[nome]
    print(f"{'OK ' if ok else 'FALHA'}  {nome:55s} origem: {ORIGEM[nome]}")


def carregar_json(nome):
    """Devolve o JSON já convertido em objetos Python (list/dict)."""
    return json.loads(EVIDENCIA[nome].decode("utf-8"))


def carregar_jsonl(nome):
    """JSONL = um JSON por linha. Devolve uma lista de dicionários."""
    return [json.loads(l) for l in EVIDENCIA[nome].decode("utf-8").splitlines() if l.strip()]


def carregar_csv(nome):
    """CSV -> lista de dicionários (cabeçalho como chaves)."""
    import csv, io
    return list(csv.DictReader(io.StringIO(EVIDENCIA[nome].decode("utf-8"))))

In [ ]:
from pathlib import Path
for nome, dados in EVIDENCIA.items():                  # recria resources/ para o script rodar como no README
    destino = Path("resources") / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(dados)
Path("lab-p6/notes").mkdir(parents=True, exist_ok=True)
print("ok")

## 2. O grafo do GCP

O script do original virou funções, para poder rodar várias vezes com cortes e com `READS` diferentes. A lógica é a mesma, linha a linha.

In [ ]:
import csv, json
from collections import deque

G = "resources/case-blueharbor/gcp/"
PROJECT = "blueharbor-analytics"
REPORT_SA = "serviceAccount:reporting-sa@blueharbor-analytics.iam.gserviceaccount.com"
GOAL = "bucket:blueharbor-customer-exports"

def montar_arestas():
    edges = []
    # 1. participação em grupos: user -> group, rótulo "member of"
    for row in csv.DictReader(open(G + "groups-export.csv")):
        edges.append((f"user:{row['member']}", f"group:{row['group_email']}", "member of"))
    # 2. vinculações no projeto, na conta de serviço e no bucket: membro -> recurso, rótulo = papel
    def bind(path, target):
        for b in json.load(open(G + path))["bindings"]:
            for m in b["members"]:
                edges.append((m, target, b["role"].removeprefix("roles/")))
    bind("iam-project-policy.json", f"project:{PROJECT}")
    bind("iam-reporting-sa-policy.json", REPORT_SA)
    bind("iam-bucket-customer-exports.json", GOAL)
    # 3. arestas de projeto que chegam a recursos específicos deste caso
    for src, dst, role in list(edges):
        if dst == f"project:{PROJECT}":
            if role in ("owner", "editor", "viewer"):      # papel básico -> membro de conveniência
                edges.append((src, f"project{role.capitalize()}:{PROJECT}", f"is project {role}"))
            if role in ("owner", "editor", "compute.instanceAdmin.v1"):   # quem administra VMs chega na VM
                edges.append((src, "vm:report-runner-1", role))
    edges.append(("vm:report-runner-1", REPORT_SA, "runs as"))   # a VM age como a conta de serviço
    return edges

def fazer_usable(reads):
    """A DECISÃO do investigador: só papéis em `reads` contam como 'ler o bucket'."""
    return lambda s, d, label: not d.startswith("bucket:") or label in reads

def caminho(edges, start, goal, usable):
    """Busca em largura (BFS): o caminho mais curto em número de saltos."""
    prev, q = {start: None}, deque([start])
    while q:
        n = q.popleft()
        if n == goal:                                   # chegou: reconstrói de trás para frente
            out = []
            while n:
                out.append(n); n = prev[n]
            return out[::-1]
        for s, d, lab in edges:
            if s == n and d not in prev and usable(s, d, lab):
                prev[d] = n; q.append(d)
    return None

def leitores(edges, goal, usable):
    """BFS ao contrário: quem consegue chegar ao alvo."""
    seen, q = {goal}, deque([goal])
    while q:
        n = q.popleft()
        for s, d, lab in edges:
            if d == n and s not in seen and usable(s, d, lab):
                seen.add(s); q.append(s)
    return sorted(x for x in seen if x.startswith(("user:", "group:", "serviceAccount:")))

def relatorio(start, edges=None, reads=("storage.objectAdmin", "storage.objectViewer"), mostrar=True):
    edges = edges if edges is not None else montar_arestas()
    usable = fazer_usable(set(reads))
    p = caminho(edges, start, GOAL, usable)
    r = leitores(edges, GOAL, usable)
    if mostrar:
        print("PATH:", " -> ".join(p) if p else "none")
        print("CAN READ OBJECTS IN THE BUCKET:")
        for x in r:
            print("  ", x)
    return p, r, edges

p_sam, r_sam, EDGES = relatorio("user:sam.reyes@vendor-example.com")
print(f"\n{len(EDGES)} arestas (esperado 36); {len(r_sam)} leitores (esperado 10)")

## 3. A decisão `READS`, por formulário

Marque se `legacyBucketReader` conta como "ler o bucket" e rode para Lee Chen.

In [ ]:
#@title Quem conta como leitor? { display-mode: "form" }
contar_legacyBucketReader = False  #@param {type:"boolean"}
identidade = "user:lee.chen@blueharbor.example"  #@param {type:"string"}

reads = {"storage.objectAdmin", "storage.objectViewer"}
if contar_legacyBucketReader:
    reads.add("storage.legacyBucketReader")           # só lista nomes; NÃO lê conteúdo
p, r, _ = relatorio(identidade, reads=reads)
print(f"\n{len(r)} leitores. Lee na lista? {'user:lee.chen@blueharbor.example' in r}")

# E com a opção invertida, para comparar na mesma rodada:
_, r2, _ = relatorio(identidade, reads=reads ^ {"storage.legacyBucketReader"}, mostrar=False)
print(f"Com a decisão invertida: {len(r2)} leitores. Lee na lista? {'user:lee.chen@blueharbor.example' in r2}")

## 4. Renderizar

O `.dot` sai do mesmo jeito que no original (caminho em vermelho, arestas que não leem em cinza tracejado). Graphviz é **opcional**: a célula de `!apt-get` só funciona no Colab; o resto do notebook não depende dela.

In [ ]:
def escrever_dot(edges, p, saida, reads=("storage.objectAdmin", "storage.objectViewer")):
    usable = fazer_usable(set(reads))
    hot = set(zip(p, p[1:])) if p else set()          # pares consecutivos do caminho
    with open(saida, "w") as f:
        f.write('digraph access {\n  rankdir=LR;\n  node [fontname="Helvetica"];\n')
        for s, d, label in edges:
            style = ", color=red, penwidth=2" if (s, d) in hot else ""
            if not usable(s, d, label):
                style = ", style=dashed, color=gray"
            f.write(f'  "{s}" -> "{d}" [label="{label}"{style}];\n')
        f.write("}\n")
    print(f"wrote {saida} with {len(edges)} edges")

escrever_dot(EDGES, p_sam, "lab-p6/notes/day-80-access-graph.dot")

In [ ]:
# (Opcional, só no Colab) instala o Graphviz e gera o PNG. Fora do Colab essas linhas são ignoradas.
!apt-get -qq install -y graphviz > /dev/null
!dot -Tpng lab-p6/notes/day-80-access-graph.dot -o lab-p6/notes/day-80-access-graph.png
import shutil
print("PNG gerado." if shutil.which("dot") and Path("lab-p6/notes/day-80-access-graph.png").exists()
      else "Sem Graphviz aqui: use o .dot no Graphviz Online ou o mermaid da próxima célula.")

In [ ]:
# Plano B sem Graphviz: a figura enxuta em mermaid (só arestas que tocam o caminho ou o bucket).
nos = {}
def nid(n):
    return nos.setdefault(n, f"N{len(nos)}")
linhas = ["graph LR"]
for s, d, lab in EDGES:
    if d == GOAL or (s, d) in set(zip(p_sam, p_sam[1:])):
        seta = "-.->" if not fazer_usable({"storage.objectAdmin", "storage.objectViewer"})(s, d, lab) else "-->"
        rot = lambda n: n.split("@")[0]                 # rótulo curto, igual para o mesmo nó
        linhas.append(f'    {nid(s)}["{rot(s)}"] {seta}|{lab}| {nid(d)}["{rot(d)}"]')
print("\n".join(linhas))

## 5. O grafo da AWS

`user/*` cobre os quatro usuários do export, então são **quatro** arestas `CreateAccessKey` (uma para o próprio `vendor-sam`).

In [ ]:
aws = carregar_json("case-blueharbor/aws/iam-authorization-details.json")
usuarios = [u["UserName"] for u in aws["UserDetailList"]]        # jq -r '.UserDetailList[].UserName'
print("Usuários:", usuarios)

AWS_EDGES = [("user:vendor-sam", f"user:{u}", "iam:CreateAccessKey (user/*)") for u in usuarios]
AWS_EDGES += [("user:svc-reporting", "s3:blueharbor-exports-mirror", "AmazonS3FullAccess"),
              ("user:dana-admin", "aws:everything", "AdministratorAccess"),
              ("user:old-etl", "aws:everything", "inline Action *"),
              ("role:MirrorWriterRole", "s3:blueharbor-exports-mirror", "s3:PutObject exports/*"),
              ("gcp:reporting-sa", "role:MirrorWriterRole", "AssumeRoleWithWebIdentity")]

# Evidência de que só UMA dessas arestas foi usada: quantos CreateAccessKey há na janela?
eventos = [json.loads(e["CloudTrailEvent"]) for e in carregar_json("case-blueharbor/aws/cloudtrail-lookup-events.json")["Events"]]
criadas = [(e["eventTime"], e["requestParameters"]["userName"]) for e in eventos if e["eventName"] == "CreateAccessKey"]
print("CreateAccessKey na janela:", criadas)
usadas = {("user:vendor-sam", f"user:{u}") for _, u in criadas} | {("user:svc-reporting", "s3:blueharbor-exports-mirror")}

with open("lab-p6/notes/day-80-aws.dot", "w") as f:
    f.write('digraph aws {\n  rankdir=LR;\n  node [fontname="Helvetica"];\n')
    for s, d, lab in AWS_EDGES:
        estilo = ", color=red, penwidth=2" if (s, d) in usadas else ""
        f.write(f'  "{s}" -> "{d}" [label="{lab}"{estilo}];\n')
    f.write("}\n")
print(Path("lab-p6/notes/day-80-aws.dot").read_text())

## 6. Testando os cortes A e B

Equivalente a copiar o script e inserir uma linha `edges = [e for e in edges if not (...)]`. Aqui a tabela sai pronta.

In [ ]:
CORTES = {
    "nenhum": lambda e: True,
    "A: data-eng -> reporting-sa (key admin)": lambda e: not (e[0] == "group:data-eng@blueharbor.example" and e[1] == REPORT_SA),
    "B: sam.reyes -> data-eng": lambda e: not (e[0] == "user:sam.reyes@vendor-example.com" and e[1] == "group:data-eng@blueharbor.example"),
}
print("| Corte | Caminho de Sam | Caminho de Priya | Leitores |\n|---|---|---|---|")
for nome, manter in CORTES.items():
    edges = [e for e in montar_arestas() if manter(e)]
    ps, rs, _ = relatorio("user:sam.reyes@vendor-example.com", edges=edges, mostrar=False)
    pp, _, _ = relatorio("user:priya.nair@blueharbor.example", edges=edges, mostrar=False)
    fmt = lambda p: f"{len(p)-1} saltos" if p else "none"
    print(f"| {nome} | {fmt(ps)} | {fmt(pp)} | {len(rs)} |")

# AWS no papel: quantas arestas CreateAccessKey cada candidato remove?
ck = [e for e in AWS_EDGES if e[2].startswith("iam:CreateAccessKey")]
print(f"\nAWS: apagar svc-reporting remove {sum(1 for e in ck if e[1] == 'user:svc-reporting')} aresta CreateAccessKey;"
      f" reescrever RotateOwnKeys para user/${{aws:username}} remove {sum(1 for e in ck if e[1] != 'user:vendor-sam')} (sobra só a de si mesmo).")

## 7. 🎮 Quiz: qual aresta cortar?

Responda **a** ou **b**.

In [ ]:
MODO_JOGO = True
QUIZ = [
    ("GCP: (a) tirar Sam de data-eng  ou  (b) tirar o serviceAccountKeyAdmin de data-eng em reporting-sa?", "b",
     "B fecha o caminho para Priya, Tom e o próximo prestador: leitores 10 -> 6, contra 10 -> 9."),
    ("AWS: (a) apagar svc-reporting  ou  (b) reescrever RotateOwnKeys para user/${aws:username}?", "b",
     "Reescrever remove 3 arestas, incluindo as não usadas para dana-admin e old-etl. Apagar svc-reporting é bom, mas deixa o caminho para o admin."),
    ("A aresta reporting-sa -> bucket remove 9 leitores. Corta? (a) sim  (b) não", "b",
     "É a função legítima do job de exportação. O algoritmo ordena, o investigador decide."),
    ("➕ Lia: (a) desativar dev.externo  ou  (b) tirar key admin de lia-devs em lia-bot?", "b",
     "Desativar a conta deixa ana.lider com o mesmo caminho até o cadastro."),
]
acertos = 0
for i, (q, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {q}")
    if MODO_JOGO:
        r = ""
        while r not in ("a", "b"):
            r = input("   a ou b? ").strip().lower()
        acertos += (r == certo); print("   ✅" if r == certo else "   ❌")
    print(f"   Gabarito: {certo} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 8. ➕ Operação Isca: o grafo da Lia

Alvo: `dataset:cadastro_espelho` (colunas com CPF, fictícias). Decisão de leitura: `bigquery.dataViewer` e `bigquery.dataOwner` leem linhas.

In [ ]:
# ------------------------------------------------------------
# DADOS DA OPERAÇÃO ISCA (fictícios) — mesmo esquema offline-first
# ------------------------------------------------------------
import base64, gzip, hashlib, json, urllib.request

BASE_ISCA = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/"
_ISCA = {
    "dia-79/lia-iam.json": "H4sIAAAAAAAC/71Vy24TMRTd5yusWYHUDKELFl01fSBVUJDKY4Mq6459M3Xw+A62J22o+B5WLPmC/hi2ZxIlLUmZtmI1flyfc+5zrgcs444Ki9key46varIebn7e/EbHJLKT8SmTxGpLU/TEHFoL8L3RQ62APZso4W9+CUXPc/aabAXBRBlXKwvhEVasRD9UUA1r0krM82wnsHVYge56wFgGjVT+kMxElS6cfQln6SJcBbaZEkkYaP2h3bkIwhYP31J56+3idbDQVH6c1+n9+Oj05B0/Ox4fZe3tj/Q5H3TL8wiaFcpIZe7osKQTSPy6F3Rp0C5EVFgVaFfIs9JSU+95NRRE9f4yYDleQVUHnBXenY0ctjE5yEqZzTxdcMZCUGP8nsRa03wo1P5ajvIQ/bzsbKG17SdlpvDyfn8D1VDizO2DRyOxAuX60RSq/NagnedTKj65bYS3HI/MBfmHuT1IiygrE2Q8cIm8fUU8wPAAvKjTf6qNSLuu7w3Ox9sT+UQBvEvdK4yPrJ+VQErw4NBzARKct8TR1agvqFcgl+UQ0d7/h45bI/x8T80/dQmm4SNINwaWNJmSPDqEcZa2UjJRT7qVoQq7pUeNEzKLbSwd3a2hRCMUdDunfAMCKFtJVtGIryFXsdZ9VXNM89/1SpXzZANRTsUUhR/3mlqPitzOvZI0liDmB8nHMwS5LafptyR8m/q9NUEbCr20TU0LiKSjPYo6trVzm42owCbbYBfuPFpDG7o/ufoIBjCQaxXc74G/uZ9uo08JKPfq750XAna+HK9uZb6uxa0t2s6rfgXRVboJewkcIsihpkays8awCJdcVhUav2yjC5ih4yXa1B5hTHEwHpO63dHuK/5yxEe7AWm0HpulyocMy406Dxql5YOUhdAOfgz+AI1eFJ3CCQAA",
}
ISCA = {}
for nome, b64 in _ISCA.items():
    embutido = gzip.decompress(base64.b64decode(b64))            # a cópia que veio com o notebook
    try:
        with urllib.request.urlopen(BASE_ISCA + nome, timeout=15) as r:
            baixado = r.read()
        # só aceita o arquivo da rede se for idêntico ao embutido (mesmo SHA-256)
        ISCA[nome] = baixado if hashlib.sha256(baixado).digest() == hashlib.sha256(embutido).digest() else embutido
        origem = "github" if ISCA[nome] is baixado else "embutida (hash diferente)"
    except Exception as erro:
        ISCA[nome], origem = embutido, f"embutida ({type(erro).__name__})"
    print(f"{nome:35s} sha256={hashlib.sha256(ISCA[nome]).hexdigest()[:16]}...  origem: {origem}")

In [ ]:
L = json.loads(ISCA["dia-79/lia-iam.json"])
LIA_SA = "serviceAccount:lia-bot@serraazul-lia.iam.gserviceaccount.example"
ALVO = "dataset:cadastro_espelho"
READS_BQ = {"bigquery.dataViewer", "bigquery.dataOwner"}
usa_bq = lambda s, d, lab: not d.startswith("dataset:") or lab in READS_BQ

def arestas_lia():
    e = [("user:" + g["membro"], "group:" + g["grupo"], "member of") for g in L["grupos"]]
    for chave, alvo in (("projeto", "project:serraazul-lia"), ("conta_de_servico_lia_bot", LIA_SA), ("dataset_cadastro_espelho", ALVO)):
        for b in L[chave]["bindings"]:
            for m in b["members"]:
                e.append((m, alvo, b["role"].removeprefix("roles/")))
    return e

CORTES_LIA = {
    "nenhum": lambda e: True,
    "A: tirar key admin de lia-devs em lia-bot": lambda e: not (e[0] == "group:lia-devs@atendemais.example" and e[1] == LIA_SA),
    "B: tirar dev.externo de lia-devs": lambda e: not (e[0] == "user:dev.externo@atendemais.example"),
}
print("| Corte | dev.externo | ana.lider | Leitores do cadastro |\n|---|---|---|---|")
for nome, manter in CORTES_LIA.items():
    E = [e for e in arestas_lia() if manter(e)]
    pd = caminho(E, "user:dev.externo@atendemais.example", ALVO, usa_bq)
    pa = caminho(E, "user:ana.lider@atendemais.example", ALVO, usa_bq)
    fmt = lambda p: f"{len(p)-1} saltos" if p else "none"
    print(f"| {nome} | {fmt(pd)} | {fmt(pa)} | {len(leitores(E, ALVO, usa_bq))} |")

E = arestas_lia()
print("\nCaminho de dev.externo:", " -> ".join(caminho(E, "user:dev.externo@atendemais.example", ALVO, usa_bq)))
print("Leitores depois do corte A:", leitores([e for e in E if CORTES_LIA["A: tirar key admin de lia-devs em lia-bot"](e)], ALVO, usa_bq))

> 💡 Depois do corte A ainda sobra `deploy-ci`, porque ele tem `serviceAccountUser` em `lia-bot` (necessário para fazer o deploy do Cloud Run). Não é para cortar às cegas: é para registrar e proteger de outro jeito (por exemplo, sem chaves gerenciadas e com aprovação no pipeline).

## ✅ Antes de fechar o Dia 80

- [ ] `day-80-access-graph.dot` (e `.png`, se tiver Graphviz) com o caminho em vermelho
- [ ] Lee Chen: `none` com o `READS` padrão; aparece com `legacyBucketReader`
- [ ] `day-80-aws.dot` com quatro `CreateAccessKey`, a usada em vermelho
- [ ] `day-80-finding.md` com a tabela de cortes (10 → 6 e 10 → 9), uma aresta por nuvem e a lacuna de pasta/organização
- [ ] `notas/dia-80-lia-achado.md` com a resposta à Atende+

**Amanhã (Dia 81):** o caminho humano até os dados: prompt injection no assistente de suporte, e a Lia sendo usada como oráculo de cadastro.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>